In [1]:
import mlflow

In [2]:
mlflow.set_experiment("Hyperparameter tuning experiment")

2026/06/14 19:16:34 INFO mlflow.tracking.fluent: Experiment with name 'Hyperparameter tuning experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/denys/PycharmProjects/Hands-On_ML/MLFLow/mlruns/2', creation_time=1781453794308, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1781453794308, lifecycle_stage='active', name='Hyperparameter tuning experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

In [9]:
import mlflow
import optuna
import sklearn


def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        rf_max_depth = trial.suggest_int("rf_max_depth", 2, 32)
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=10)
        rf_max_features = trial.suggest_float("rf_max_features", 0.2, 1.0)
        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
            "max_features": rf_max_features,
        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = sklearn.ensemble.RandomForestRegressor(**params)
        regressor_obj.fit(X_train, y_train)

        y_pred = regressor_obj.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        # Log current trial's error metric
        mlflow.log_metrics({"error": error})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error

In [10]:
 # Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-06-14 19:30:05,557] A new study created in memory with name: no-name-018b511c-514d-45ad-8b60-2262048867fc
2026/06/14 19:30:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-14 19:30:09,421] Trial 0 finished with value: 0.7368793934196417 and parameters: {'rf_max_depth': 2, 'rf_n_estimators': 280, 'rf_max_features': 0.6198569268593028}. Best is trial 0 with value: 0.7368793934196417.
2026/06/14 19:30:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization

In [11]:
mlflow.register_model(
    model_uri="runs:/72cd9676bf75458f96bdd47a57cee1d2/model",
    name="housing-price-predictor",
)

Successfully registered model 'housing-price-predictor'.
2026/06/14 19:45:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 72cd9676bf75458f96bdd47a57cee1d2 has no artifacts at artifact path 'model', registering model based on models:/m-b7f9b66593ba42caa7d3f6cf0fc56ac7 instead
Created version '1' of model 'housing-price-predictor'.


<ModelVersion: aliases=[], creation_timestamp=1781455510236, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1781455510236, metrics=None, model_id=None, name='housing-price-predictor', params=None, run_id='72cd9676bf75458f96bdd47a57cee1d2', run_link=None, source='models:/m-b7f9b66593ba42caa7d3f6cf0fc56ac7', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>